# Controller validation + nucleus ablation analysis

This notebook analyzes validation evaluations from `run_controller_validation_eval.sh` and nucleus-insertion ablations from `run_controller_nucleus_ablation.sh`.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "analysis":
    PROJECT_ROOT = PROJECT_ROOT.parent

RUN_ROOT = PROJECT_ROOT / "outputs" / "controller_sft_m3cot_test_sweeps"
EVAL_ROOT = RUN_ROOT / "eval_validation"
ABLATION_ROOT = RUN_ROOT / "nucleus_ablation_validation"

print("RUN_ROOT:", RUN_ROOT)
print("EVAL_ROOT:", EVAL_ROOT)
print("ABLATION_ROOT:", ABLATION_ROOT)

In [ ]:
def load_summary(path: Path):
    with open(path, "r", encoding="utf-8") as handle:
        data = json.load(handle)
    metrics = data.get("metrics", {})
    nucleus = data.get("nucleus_insertion", {})
    return {
        "summary_path": str(path),
        "num_examples": data.get("num_examples"),
        "dataset_partition": data.get("dataset_partition"),
        "controller_checkpoint": data.get("controller_checkpoint"),
        "accuracy": metrics.get("accuracy"),
        "correct": metrics.get("correct"),
        "total": metrics.get("total"),
        "avg_controller_actions": metrics.get("avg_controller_actions"),
        "avg_controller_tokens": metrics.get("avg_controller_tokens"),
        "avg_output_tokens": metrics.get("avg_output_tokens"),
        "avg_total_tokens": metrics.get("avg_total_tokens"),
        "trace_attention_mass": metrics.get("trace_attention_mass"),
        "visual_trace_attention_mass": metrics.get("visual_trace_attention_mass"),
        "think_attention_mass": metrics.get("think_attention_mass"),
        "nucleus_enabled": nucleus.get("enabled"),
        "nucleus_scope": nucleus.get("scope"),
        "nucleus_top_p": nucleus.get("top_p"),
        "nucleus_max_indices": nucleus.get("max_indices"),
    }

def discover_eval_summaries(root: Path, mode: str) -> pd.DataFrame:
    rows = []
    for path in root.glob("**/*_summary.json"):
        rel = path.relative_to(root).parts
        if mode == "plain" and len(rel) >= 3:
            trace_source, variant = rel[0], rel[1]
            ablation = "off"
        elif mode == "ablation" and len(rel) >= 4:
            trace_source, variant, ablation = rel[0], rel[1], rel[2]
        else:
            continue
        row = load_summary(path)
        row.update({"mode": mode, "trace_source": trace_source, "variant": variant, "ablation": ablation})
        rows.append(row)
    return pd.DataFrame(rows)

plain_df = discover_eval_summaries(EVAL_ROOT, "plain")
ablation_df = discover_eval_summaries(ABLATION_ROOT, "ablation")

display(plain_df.sort_values(["trace_source", "variant"]) if not plain_df.empty else plain_df)
display(ablation_df.sort_values(["trace_source", "variant", "ablation"]) if not ablation_df.empty else ablation_df)

In [ ]:
if not plain_df.empty:
    cols = ["accuracy", "avg_controller_actions", "avg_controller_tokens", "avg_output_tokens", "avg_total_tokens"]
    display(plain_df.pivot_table(index="variant", columns="trace_source", values=cols))

    ax = plain_df.pivot_table(index="variant", columns="trace_source", values="accuracy").plot(kind="bar", figsize=(10, 4))
    ax.set_title("Validation accuracy by trained controller")
    ax.set_ylabel("accuracy")
    ax.set_ylim(0, max(1.0, (plain_df["accuracy"].max() or 0) * 1.1))
    plt.tight_layout()
    plt.show()

In [ ]:
if not ablation_df.empty:
    ordered = ["off", "patch_only", "region_only"]
    ablation_df["ablation"] = pd.Categorical(ablation_df["ablation"], ordered, ordered=True)
    display(ablation_df.pivot_table(index=["trace_source", "variant"], columns="ablation", values="accuracy"))

    for trace_source, sub in ablation_df.groupby("trace_source"):
        ax = sub.pivot_table(index="variant", columns="ablation", values="accuracy").plot(kind="bar", figsize=(10, 4))
        ax.set_title(f"Nucleus insertion ablation: {trace_source}")
        ax.set_ylabel("validation accuracy")
        ax.set_ylim(0, max(1.0, (sub["accuracy"].max() or 0) * 1.1))
        plt.tight_layout()
        plt.show()

In [ ]:
if not ablation_df.empty:
    base = ablation_df[ablation_df["ablation"] == "off"][["trace_source", "variant", "accuracy"]].rename(columns={"accuracy": "accuracy_off"})
    delta_df = ablation_df.merge(base, on=["trace_source", "variant"], how="left")
    delta_df["accuracy_delta_vs_off"] = delta_df["accuracy"] - delta_df["accuracy_off"]
    display(delta_df.sort_values(["trace_source", "variant", "ablation"])[[
        "trace_source", "variant", "ablation", "accuracy", "accuracy_off", "accuracy_delta_vs_off",
        "avg_controller_actions", "avg_controller_tokens", "avg_total_tokens"
    ]])

In [ ]:
def read_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                yield json.loads(line)

def predictions_for_summary(summary_path: str) -> Path:
    path = Path(summary_path)
    return path.with_name(path.name.replace("_summary.json", ".jsonl"))

# Optional: inspect per-example correctness flips between ablations for one run.
# Set these to any row values from ablation_df.
TRACE_SOURCE = None  # e.g. "global_test"
VARIANT = None       # e.g. "multihot_binary"

if TRACE_SOURCE and VARIANT and not ablation_df.empty:
    selected = ablation_df[(ablation_df.trace_source == TRACE_SOURCE) & (ablation_df.variant == VARIANT)]
    per_ablation = {}
    for row in selected.to_dict("records"):
        pred_path = predictions_for_summary(row["summary_path"])
        if pred_path.exists():
            per_ablation[str(row["ablation"])] = pd.DataFrame(read_jsonl(pred_path))[['example_id', 'correct', 'generated_text']]
    if per_ablation:
        merged = None
        for name, df in per_ablation.items():
            df = df.rename(columns={"correct": f"correct_{name}", "generated_text": f"generated_text_{name}"})
            merged = df if merged is None else merged.merge(df, on="example_id", how="outer")
        display(merged)

## Reading the ablation

`off` is the standard one-visual-index-at-a-time controller. `patch_only` lets a selected PATCH action insert a top-p set of patches in one controller step. `region_only` does the same for REGION. The delta table shows whether the extra visual evidence helps accuracy and how much it changes action/token budgets.